In [ ]:
import os
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

# ==========================================
# 1. DATASET GENERATION
# ==========================================

def generate_dataset(num_samples=100, img_size=64):
    images = []
    labels = []  # 0: circle, 1: triangle, 2: rectangle
    label_map = {0: 'Circle', 1: 'Triangle', 2: 'Rectangle'}

    for i in range(num_samples):
        img = np.zeros((img_size, img_size), dtype=np.uint8)
        shape_type = np.random.choice(['circle', 'triangle', 'rectangle'])

        if shape_type == 'circle':
            center = (np.random.randint(15, img_size - 15), np.random.randint(15, img_size - 15))
            radius = np.random.randint(8, 18)
            cv2.circle(img, center, radius, 255, -1)
            labels.append(0)

        elif shape_type == 'triangle':
            pt1 = [np.random.randint(10, img_size // 2), np.random.randint(10, img_size // 2)]
            pt2 = [np.random.randint(img_size // 2, img_size - 10), np.random.randint(10, img_size // 2)]
            pt3 = [np.random.randint(10, img_size - 10), np.random.randint(img_size // 2, img_size - 10)]
            pts = np.array([pt1, pt2, pt3], np.int32)
            cv2.fillPoly(img, [pts], 255)
            labels.append(1)

        elif shape_type == 'rectangle':
            w = np.random.randint(20, 40)
            h = np.random.randint(10, 20)
            if np.random.rand() > 0.5: w, h = h, w
            x = np.random.randint(5, img_size - w - 5)
            y = np.random.randint(5, img_size - h - 5)
            cv2.rectangle(img, (x, y), (x + w, y + h), 255, -1)
            labels.append(2)

        images.append(img)

    return np.array(images), np.array(labels), label_map

# Generate samples
X, y, label_names = generate_dataset(num_samples=10000)


# ==========================================
# 2. DISK IO (SAVING TRAIN & TEST DATA)
# ==========================================

base_dir = 'dataset'
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

# Ensure we use the exact labels from the generation script
label_names = {0: 'Circle', 1: 'Triangle', 2: 'Rectangle'}

for label in label_names.values():
    os.makedirs(os.path.join(base_dir, 'train', label), exist_ok=True)

test_dir = os.path.join(base_dir, 'test_single')
os.makedirs(test_dir, exist_ok=True)

# Save Training Data (0-999)
for i in range(1000):
    label_text = label_names[y[i]]
    img = Image.fromarray(X[i])
    img.save(os.path.join(base_dir, 'train', label_text, f'{i}.png'))

# Save Test Data (1000-1999)
test_indices = np.arange(1000, 2000)
labels_for_csv = []
for i in range(1000):
    source_idx = test_indices[i]
    label_text = label_names[y[source_idx]]
    img = Image.fromarray(X[source_idx])

    img.save(os.path.join(test_dir, f'test_{i}.png'))
    labels_for_csv.append(label_text)

# Save CSV
with open('test_labels.csv', 'w') as f:
    for lbl in labels_for_csv:
        f.write(f"{lbl}\n")

print(f"Dataset reset and saved. test_labels.csv created with {len(labels_for_csv)} rows.")


# ==========================================
# 3. PYTORCH DATASETS & MODEL DEFINITION
# ==========================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class TestDataset(Dataset):
    def __init__(self, img_dir, labels_csv, mapping, transform=None):
        self.img_dir = img_dir
        with open(labels_csv, 'r') as f:
            self.labels = [line.strip() for line in f.readlines()]
        self.transform = transform
        self.mapping = mapping  # Receives the training set's structural mapping

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f'test_{idx}.png')
        image = Image.open(img_path)
        label_name = self.labels[idx]

        # Pull index directly from training map rules (Alphabetical)
        label = self.mapping[label_name]

        if self.transform:
            image = self.transform(image)
        return image, label

# Simple CNN Model
class ShapeCNN(nn.Module):
    def __init__(self):
        super(ShapeCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 128), nn.ReLU(),
            nn.Linear(128, 3)
        )
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# ==========================================
# 4. TRAINING & EVALUATION
# ==========================================

# 1. Let ImageFolder instantiate naturally (Alphabetical sorting: Circle:0, Rectangle:1, Triangle:2)
train_dataset = datasets.ImageFolder(root='dataset/train', transform=transform)

# 2. Extract that exact target rule map from the training dataset
active_training_mapping = train_dataset.class_to_idx
print(f"Active training map rules enforced by PyTorch: {active_training_mapping}")

# 3. Pass that exact active rule map down to your custom evaluation class
test_dataset = TestDataset(
    img_dir='dataset/test_single',
    labels_csv='test_labels.csv',
    mapping=active_training_mapping,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = ShapeCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting Training...")
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/10, Loss: {running_loss/len(train_loader):.4f}")

# Final Evaluation
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nFinal Accuracy on the randomized test set: {100 * correct / total:.2f}%")

Dataset reset and saved. test_labels.csv created with 1000 rows.
Active training map rules enforced by PyTorch: {'Circle': 0, 'Rectangle': 1, 'Triangle': 2}
Starting Training...
Epoch 1/10, Loss: 0.9552
Epoch 2/10, Loss: 0.7157
Epoch 3/10, Loss: 0.5196
Epoch 4/10, Loss: 0.3351
Epoch 5/10, Loss: 0.1552
Epoch 6/10, Loss: 0.0593
Epoch 7/10, Loss: 0.0283
Epoch 8/10, Loss: 0.0147
Epoch 9/10, Loss: 0.0149
Epoch 10/10, Loss: 0.0067

Final Accuracy on the randomized test set: 98.90%


In [ ]:
!zip -r downloaded_folder.zip /content/dataset

In [3]:
!nvidia-smi

Wed Jun  3 13:18:01 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 537.70                 Driver Version: 537.70       CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   41C    P8               3W /  70W |    355MiB /  6141MiB |     40%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--